# Geração de Energia Renovável — Solar e Eólica

**Capítulo aplicado:** 8 (Fontes renováveis).

Aplica as **fórmulas físicas** exatas do capítulo:

Potência do vento (eólica):
$$P = \\tfrac{1}{2} \\rho A v^3$$

Limite de Betz (constante teórica):
$$C_{p,\\max} = \\tfrac{16}{27} \\approx 0{,}593$$

Energia solar:
$$E = \\text{irradiância} \\times A \\times \\eta$$

Mais a **curva de operação** completa com cut-in / nominal / cut-out.

## 1. Constante de Betz

In [ ]:
BETZ = 16 / 27   # limite teorico do capitulo (~0.593)
print("Betz =", round(BETZ, 4))

## 2. Potência eólica com curva de operação

Curva do capítulo:
- $v < $ cut-in → P = 0;
- cut-in ≤ $v <$ nominal → P cresce com $v^3$;
- nominal ≤ $v <$ cut-out → P saturada (= $P_\\text{nominal}$);
- $v ≥$ cut-out → P = 0 (turbina desliga).

In [ ]:
def potencia_eolica(velocidade, rho, area, cp,
                    cut_in, v_nominal, cut_out):
    if cp > BETZ:
        cp = BETZ

    if velocidade < cut_in:
        return 0.0
    if velocidade >= cut_out:
        return 0.0

    p = 0.5 * rho * area * (velocidade ** 3) * cp
    p_nominal = 0.5 * rho * area * (v_nominal ** 3) * cp
    if p > p_nominal:
        p = p_nominal
    return p


def classificar_operacao_turbina(velocidade, cut_in, v_nominal, cut_out):
    if velocidade < cut_in:
        return "PARADA (vento fraco)"
    elif velocidade < v_nominal:
        return "GERANDO (regiao cubica)"
    elif velocidade < cut_out:
        return "NOMINAL (saturada)"
    else:
        return "DESLIGADA (vento forte)"

## 3. Geração solar e escolha de painel

$E = \\text{irradiância} \\times A \\times \\eta$

Tipos de células do capítulo:
- monocristalino: η ~ 0.20 (maior eficiência, caro);
- policristalino: η ~ 0.15 (custo-benefício);
- filme fino: η ~ 0.10 (leve e flexível).

In [ ]:
def potencia_solar(irradiancia, area, eficiencia):
    if irradiancia <= 0 or area <= 0 or eficiencia <= 0:
        return 0.0
    return irradiancia * area * eficiencia


def escolher_tipo_painel(prioridade):
    """Selecao via if/elif (cap. 3) com dados do cap. 8."""
    if prioridade == "eficiencia":
        return {"tipo": "monocristalino", "eficiencia": 0.20,
                "obs": "Maior eficiencia, custo alto."}
    elif prioridade == "custo":
        return {"tipo": "policristalino", "eficiencia": 0.15,
                "obs": "Custo-beneficio intermediario."}
    elif prioridade == "peso":
        return {"tipo": "filme_fino", "eficiencia": 0.10,
                "obs": "Leve e flexivel, eficiencia menor."}
    else:
        return {"tipo": "policristalino", "eficiencia": 0.15,
                "obs": "Padrao."}

## 4. Demonstração — curva de operação eólica em Marte

Densidade do ar em Marte: ~0.020 kg/m³ (muito menor que na Terra).

In [ ]:
print("Velocidade | Estado                    | Potencia (W)")
print("-----------+---------------------------+-------------")
for v in [0, 2, 5, 10, 12, 15, 20, 26]:
    estado = classificar_operacao_turbina(v, 3, 12, 25)
    p = potencia_eolica(v, rho=0.020, area=12, cp=0.40,
                        cut_in=3, v_nominal=12, cut_out=25)
    print(f"  {v:>5} m/s | {estado:25s} | {p:>10.3f}")

print()
print("Escolha de painel solar:")
for pri in ["eficiencia", "custo", "peso"]:
    print(f"  prioridade={pri:11s} ->", escolher_tipo_painel(pri))

print()
print("Energia solar (area=20 m2, irradiancia=360 W/m2, mono):")
print("  P =", potencia_solar(360, 20, 0.20), "W")